In [18]:
import os
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer



In [19]:
raw_data = "../data/raw/appendicitis_comprehensive_dataset.csv "
df = pd.read_csv(raw_data)

print(df.shape)
display(df.head(10))
display(df.info())


(1500, 23)


,Patient_ID,Age,Gender,BMI,Is_Pregnant,Duration_of_Symptoms_Hours,Pain_Migration,Abdominal_Pain_Location,Nausea_Vomiting,Loss_of_Appetite,Fever_Temp_C,Rebound_Tenderness,McBurney_Sign,Rovsing_Sign,Psoas_Sign,WBC_Count_k_uL,Neutrophil_Percentage,CRP_Level_mg_L,Ultrasound_Findings,Pathological_Cause,Severity,Management,Final_Diagnosis
0,APX-0001,32,Female,23.4,No,17,Yes,RLQ,Yes,No,37.7,Yes,Yes,Yes,Yes,11.5,77.8,63.2,Non-visualized,Fecalith/Appendicolith,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
1,APX-0002,5,Male,19.4,No,31,No,RLQ,No,No,38.2,Yes,Yes,Yes,No,12.6,74.7,89.1,Target Sign,Lymphoid Hyperplasia,Uncomplicated,Antibiotics (Conservative),Appendicitis
2,APX-0003,28,Female,27.5,No,49,Yes,RLQ,Yes,Yes,37.7,Yes,Yes,No,Yes,16.2,77.0,92.9,Periappendiceal Fluid,Lymphoid Hyperplasia,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
3,APX-0004,20,Male,28.7,No,32,No,RLQ,Yes,Yes,37.5,Yes,Yes,Yes,Yes,19.1,88.3,12.7,Periappendiceal Fluid,Fecalith/Appendicolith,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
4,APX-0005,12,Female,22.0,No,47,No,RLQ,No,No,37.3,No,No,Yes,No,12.2,65.0,8.8,Normal,NaN,NaN,Observation/Other,Other (Ectopic Pregnancy (if pregnant))
5,APX-0006,21,Female,20.4,No,43,Yes,RLQ,Yes,Yes,37.5,Yes,No,Yes,No,11.3,81.5,34.8,Appendicolith Seen,Lymphoid Hyperplasia,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
6,APX-0007,57,Male,28.9,No,19,Yes,Generalized,No,Yes,38.7,Yes,Yes,No,No,11.2,78.8,28.0,Enlarged (>6mm),Unknown,Uncomplicated,Laparoscopic Appendectomy,Appendicitis
7,APX-0008,41,Female,28.7,No,28,Yes,Generalized,Yes,Yes,38.5,Yes,No,Yes,Yes,16.9,87.5,70.6,Periappendiceal Fluid,Unknown,Complicated (Perforated/Gangrenous),Open Appendectomy,Appendicitis
8,APX-0009,21,Female,26.3,No,8,Yes,Generalized,Yes,Yes,37.9,Yes,Yes,Yes,No,14.4,82.9,199.8,Target Sign,Unknown,Uncomplicated,Antibiotics (Conservative),Appendicitis
9,APX-0010,5,Female,20.6,No,29,No,RLQ,Yes,Yes,38.0,No,Yes,No,Yes,20.9,79.1,42.0,Enlarged (>6mm),Lymphoid Hyperplasia,Uncomplicated,Laparoscopic Appendectomy,Appendicitis


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Patient_ID                  1500 non-null   object 
 1   Age                         1500 non-null   int64  
 2   Gender                      1500 non-null   object 
 3   BMI                         1500 non-null   float64
 4   Is_Pregnant                 1500 non-null   object 
 5   Duration_of_Symptoms_Hours  1500 non-null   int64  
 6   Pain_Migration              1500 non-null   object 
 7   Abdominal_Pain_Location     1500 non-null   object 
 8   Nausea_Vomiting             1500 non-null   object 
 9   Loss_of_Appetite            1500 non-null   object 
 10  Fever_Temp_C                1500 non-null   float64
 11  Rebound_Tenderness          1500 non-null   object 
 12  McBurney_Sign               1500 non-null   object 
 13  Rovsing_Sign                1500 

None

In [20]:
target_col = "Final_Diagnosis"

drop_cols = [
    "Patient_ID",
    "Management",
    "Pathological_Cause",
    "Severity"
]

drop_col = [c for c in drop_cols if c in df.columns]
print("Dropping columns:", drop_col)

Dropping columns: ['Patient_ID', 'Management', 'Pathological_Cause', 'Severity']


In [21]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=drop_col)
df_clean["target_bin"] = (df_clean[target_col] == "Appendicitis").astype(int)

display(df_clean[[target_col, "target_bin"]].head())
print(df_clean["target_bin"].value_counts())


boolean_cols = ['Is_Pregnant', "Pain_Migration", "Nausea_Vomiting", "Loss_of_Appetite", "Rebound_Tenderness", "McBurney_Sign", "Rovsing_Sign", "Psoas_Sign"]

for col in boolean_cols:
    if col in df_clean.columns:
        df_clean[col] = (df_clean[col].astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0})).fillna(0).astype(int)
display(df_clean[boolean_cols].head())

,Final_Diagnosis,target_bin
0,Appendicitis,1
1,Appendicitis,1
2,Appendicitis,1
3,Appendicitis,1
4,Other (Ectopic Pregnancy (if pregnant)),0


target_bin
1    1190
0     310
Name: count, dtype: int64


,Is_Pregnant,Pain_Migration,Nausea_Vomiting,Loss_of_Appetite,Rebound_Tenderness,McBurney_Sign,Rovsing_Sign,Psoas_Sign
0,0,1,1,0,1,1,1,1
1,0,0,0,0,1,1,1,0
2,0,1,1,1,1,1,0,1
3,0,0,1,1,1,1,1,1
4,0,0,0,0,0,0,1,0


In [22]:
X = df_clean.drop(columns=[target_col, "target_bin"], errors="ignore")
y = df_clean["target_bin"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)



Features shape: (1500, 18)
Target shape: (1500,)


In [23]:
numeric_feat = X.select_dtypes(include=[np.number]).columns.tolist()
categorcal_feat = X.select_dtypes(include=["object", "category"]).columns.tolist()

display(X[numeric_feat].head())
display(X[categorcal_feat].head())

,Age,BMI,Is_Pregnant,Duration_of_Symptoms_Hours,Pain_Migration,Nausea_Vomiting,Loss_of_Appetite,Fever_Temp_C,Rebound_Tenderness,McBurney_Sign,Rovsing_Sign,Psoas_Sign,WBC_Count_k_uL,Neutrophil_Percentage,CRP_Level_mg_L
0,32,23.4,0,17,1,1,0,37.7,1,1,1,1,11.5,77.8,63.2
1,5,19.4,0,31,0,0,0,38.2,1,1,1,0,12.6,74.7,89.1
2,28,27.5,0,49,1,1,1,37.7,1,1,0,1,16.2,77.0,92.9
3,20,28.7,0,32,0,1,1,37.5,1,1,1,1,19.1,88.3,12.7
4,12,22.0,0,47,0,0,0,37.3,0,0,1,0,12.2,65.0,8.8


,Gender,Abdominal_Pain_Location,Ultrasound_Findings
0,Female,RLQ,Non-visualized
1,Male,RLQ,Target Sign
2,Female,RLQ,Periappendiceal Fluid
3,Male,RLQ,Periappendiceal Fluid
4,Female,RLQ,Normal


In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)
print("Train Class balance: ", y_train.value_counts(normalize=True))
print("Test Class balance: ", y_test.value_counts(normalize=True))

Training set shape: (1200, 18) (1200,)
Test set shape: (300, 18) (300,)
Train Class balance:  target_bin
1    0.793333
0    0.206667
Name: proportion, dtype: float64
Test Class balance:  target_bin
1    0.793333
0    0.206667
Name: proportion, dtype: float64


In [25]:
numeric_pipeline =Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))

])


preprocessor = ColumnTransformer(
    transformers = [
        ("num", numeric_pipeline, numeric_feat),
        ("cat", categorical_pipeline, categorcal_feat)
    ]
)


X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training features shape:", X_train_processed.shape)
print("Processed test features shape:", X_test_processed.shape)

Processed training features shape: (1200, 32)
Processed test features shape: (300, 32)


In [26]:
os.makedirs("../data/interim", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

df_clean.to_csv("../data/interim/appendicitis_clean.csv", index=False)


feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed.toarray() if hasattr(X_test_processed, 'toarray') else X_test_processed, columns=feature_names, index=X_test.index)

X_train_df.to_csv("../data/processed/X_train.csv", index=True)
X_test_df.to_csv("../data/processed/X_test.csv", index=True)
y_train.to_csv("../data/processed/y_train.csv", index=True)
y_test.to_csv("../data/processed/y_test.csv", index=True)


print("Data cleaning and preprocessing complete. Processed files saved to ../data/processed/")


Data cleaning and preprocessing complete. Processed files saved to ../data/processed/


In [27]:
print("Any nulls in cleaned interim:", df_clean.isna().sum().sum())
print("Any nulls in X_train processed:", X_train_df.isna().sum().sum())
print("Any nulls in X_test processed:", X_test_df.isna().sum().sum())

Any nulls in cleaned interim: 0
Any nulls in X_train processed: 0
Any nulls in X_test processed: 0
